# Fast3R 复现：环境检查

运行下面的代码单元，确认 Notebook 使用的是 `fast3r` Conda 环境，并检查 Python、PyTorch、CUDA、GPU、Fast3R 和关键依赖。

In [3]:
import os
import platform
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path


def package_version(package_name):
    try:
        return version(package_name)
    except PackageNotFoundError:
        return "未安装"


print("=== Python 环境 ===")
print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
print("Conda 环境:", os.environ.get("CONDA_DEFAULT_ENV", "未检测到"))
print("系统:", platform.platform())

print("\n=== PyTorch / CUDA ===")
try:
    import torch

    print("PyTorch:", torch.__version__)
    print("Torch CUDA:", torch.version.cuda)
    print("CUDA 可用:", torch.cuda.is_available())
    print("GPU 数量:", torch.cuda.device_count())
    if torch.cuda.is_available():
        for index in range(torch.cuda.device_count()):
            properties = torch.cuda.get_device_properties(index)
            memory_gb = properties.total_memory / 1024**3
            print(f"GPU {index}: {properties.name} ({memory_gb:.1f} GB)")
except ImportError as error:
    print("PyTorch 导入失败:", error)

print("\n=== Fast3R / 关键依赖 ===")
for package_name in [
    "fast3r",
    "numpy",
    "open3d",
    "jupyterlab",
    "ipykernel",
    "gradio",
    "lightning",
    "huggingface-hub",
]:
    print(f"{package_name}: {package_version(package_name)}")

print("\n=== 项目路径 ===")
project_root = Path.cwd()
print("当前目录:", project_root)
for relative_path in [
    "fast3r",
    "checkpoints/Fast3R_ViT_Large_512",
    "demo_examples",
    "requirements.txt",
]:
    path = project_root / relative_path
    print(f"{relative_path}: {'存在' if path.exists() else '缺失'}")

print("\n=== Fast3R 导入检查 ===")
try:
    import fast3r
    from fast3r.dust3r.inference_multiview import inference
    from fast3r.models.fast3r import Fast3R

    print("Fast3R 导入: 成功")
    print("Fast3R 路径:", fast3r.__file__)
except Exception as error:
    print("Fast3R 导入失败:", repr(error))

=== Python 环境 ===
Python: 3.11.16
解释器: /home/yyz/miniconda3/envs/fast3r/bin/python
Conda 环境: fast3r
系统: Linux-6.8.0-138-generic-x86_64-with-glibc2.35

=== PyTorch / CUDA ===
PyTorch: 2.7.1+cu128
Torch CUDA: 12.8
CUDA 可用: True
GPU 数量: 1
GPU 0: NVIDIA GeForce RTX 5070 Ti Laptop GPU (11.5 GB)

=== Fast3R / 关键依赖 ===
fast3r: 1.0
numpy: 1.26.4
open3d: 0.19.0
jupyterlab: 4.6.3
ipykernel: 7.3.0
gradio: 5.23.0
lightning: 2.6.5
huggingface-hub: 1.30.0

=== 项目路径 ===
当前目录: /home/yyz/fast3r
fast3r: 存在
checkpoints/Fast3R_ViT_Large_512: 存在
demo_examples: 存在
requirements.txt: 存在

=== Fast3R 导入检查 ===
Fast3R 导入: 成功
Fast3R 路径: /home/yyz/fast3r/fast3r/__init__.py


## 在 Notebook 中启动 Gradio Demo

运行下面的代码单元后，模型和 Gradio 服务会在后台启动。请在浏览器中打开输出的本地地址完成上传和可视化。

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

project_root = Path.cwd()
checkpoint_dir = project_root / "checkpoints" / "Fast3R_ViT_Large_512"
demo_script = project_root / "fast3r" / "viz" / "demo.py"

demo_process = globals().get("demo_process")
if demo_process is not None and demo_process.poll() is None:
    print(f"Demo 已在运行，PID: {demo_process.pid}")
    print("浏览器地址: http://127.0.0.1:7860")
else:
    import torch

    can_start = True
    if torch.cuda.is_available():
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        free_gb = free_bytes / 1024**3
        print(f"GPU 可用显存: {free_gb:.2f} / {total_bytes / 1024**3:.2f} GB")
        if free_gb < 4.0:
            can_start = False
            print("暂不启动 Demo：可用显存不足 4 GB。")
            print("请先停止其他 GPU 训练任务，再重新运行本单元。")

    if can_start:
        if not checkpoint_dir.exists():
            raise FileNotFoundError(f"找不到模型目录: {checkpoint_dir}")
        if not demo_script.exists():
            raise FileNotFoundError(f"找不到 Demo 脚本: {demo_script}")

        environment = os.environ.copy()
        environment["GRADIO_TEMP_DIR"] = str(project_root / "gradio_tmp_dir")
        demo_process = subprocess.Popen(
            [
                sys.executable,
                str(demo_script),
                "--checkpoint_dir",
                str(checkpoint_dir),
                "--examples_dir",
                str(project_root / "demo_examples"),
                "--output_dir",
                str(project_root / "demo_outputs"),
            ],
            cwd=project_root,
            env=environment,
            start_new_session=True,
        )
        time.sleep(2)
        if demo_process.poll() is not None:
            raise RuntimeError(f"Demo 启动失败，退出码: {demo_process.returncode}")

        print(f"Demo 已在后台启动，PID: {demo_process.pid}")
        print("请在浏览器打开: http://127.0.0.1:7860")
        print("如果端口被占用，请先运行停止单元，再重新启动。")

GPU 可用显存: 9.65 / 11.47 GB
Demo 已在后台启动，PID: 155946
请在浏览器打开: http://127.0.0.1:7860
如果端口被占用，请先运行停止单元，再重新启动。


Warning, cannot find cuda-compiled version of RoPE2D, using a slow pytorch version instead


/home/yyz/miniconda3/envs/fast3r/lib/python3.11/site-packages/pl_bolts/__init__.py:11: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(numpy, tp_name):
/home/yyz/miniconda3/envs/fast3r/lib/python3.11/site-packages/pl_bolts/__init__.py:11: FutureWarning: In the future `np.bool` will be defined as the corresponding NumPy scalar.
  if not hasattr(numpy, tp_name):
/home/yyz/miniconda3/envs/fast3r/lib/python3.11/site-packages/lightning_fabric/__init__.py:29: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
/home/yyz/miniconda3/envs/fast3r/lib/python3.11/site-packages/pl_bolts/models/self_supervised/amdim/amdim_module.py:34: UnderReviewWarning: The feature generate_power_seq is currently marked under review. The compatibility with other Lightning projects

[13:21:02] ViserServerManager started                                 ]8;id=5371292;file:///home/yyz/fast3r/fast3r/viz/demo.py\demo.py]8;;\:]8;id=5371293;file:///home/yyz/fast3r/fast3r/viz/demo.py#77\77]8;;\
Loading weights from local directory
* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.
Extracted 9 frames from video using OpenCV
>> Loading a list of 9 images
 - adding /home/yyz/fast3r/gradio_tmp_dir/fe467f95460a11e742346fca7534409eeaa5b1085c11fa7175c8c045404d2ac8/frame_000000.jpg with resolution 1080x1920 --> 224x224
 - adding /home/yyz/fast3r/gradio_tmp_dir/e4f6520fa4467711fb9627d138d182d584dec1ac74d1a99e1bd9f406a6defb44/frame_000001.jpg with resolution 1080x1920 --> 224x224
 - adding /home/yyz/fast3r/gradio_tmp_dir/445cc20fdba3f402684f0abaad49372758abb7e3756549bf6ce524d22ae91fb2/frame_000002.jpg with resolution 1080x1920 --> 224x224
 - adding /home/yyz/fast3r/gradio_tmp_dir/f962ce1cf64570c236da6bf0d9a4341e10e12acbc

100%|██████████| 9/9 [00:00<00:00, 486.47it/s]


(viser) Share URL requested!
(viser) Generated share URL (expires in 24 hours, max 16 clients): 
https://abstract-visual.share.viser.studio
[13:22:13] Server 1 launched with URL                                ]8;id=5371306;file:///home/yyz/fast3r/fast3r/viz/demo.py\demo.py]8;;\:]8;id=5371307;file:///home/yyz/fast3r/fast3r/viz/demo.py#127\127]8;;\
           https://abstract-visual.share.viser.studio (pid: 157691)             
           for session session_1789536130877                                    
(viser) Connection opened (0, 1 total), 126 persistent messages
(viser) Connection closed (0, 0 total)


## 停止 Notebook 启动的 Demo

在浏览器中完成观察后，运行下面的单元释放模型占用的显存。

In [11]:
demo_process = globals().get("demo_process")
if demo_process is not None and demo_process.poll() is None:
    demo_process.terminate()
    try:
        demo_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        demo_process.kill()
        demo_process.wait()
    print(f"Demo 已停止，PID: {demo_process.pid}")
    demo_process = None
else:
    print("当前没有由本 Notebook 启动的 Demo 进程。")

Demo 已停止，PID: 155946
